In [ ]:
from functools import partial

from mxlpy import Scipy

from mxlmodels import Simulator, get_matuszynska2019, plot

res = (
    Simulator(
        get_matuszynska2019(),
        integrator=partial(Scipy, method="Radau"),
    )
    .simulate(10)
    .get_result()
    .unwrap_or_err()
)

fig, (ax1, ax2) = plot.two_axes(figsize=(12, 6))
_ = plot.lines(res.variables.iloc[:, :10], ax=ax1)
_ = plot.lines(res.fluxes.iloc[:, :10], ax=ax2)

## Reproduce Fig. 2 from the original paper

Original caption reads: 'Simulations of light–dark–light transitions for different light intensities, ranging from 20 to 200 μmol m−2 s−1. Shown are the dynamics of internal orthophosphate concentration, triose phosphate transporter (TPT) export and carbon fixation rates. The simulated time‐courses are shown from 200 s, when the system has reached a stationary state. From 300 to 500 s (gray area), the external light has been set to 5 μmol m−2 s−1. The figure illustrates that for low light intensities the CBB cycle fails to restart in the second light period.'

In [ ]:
import matplotlib.pyplot as plt


def LDL(m, Tmax, intensity, darkphase):
    """Simple Light-Dark-Light simulation"""

    s = Simulator(
        m,
        integrator=partial(Scipy, method="Radau", atol=1e-14),
    )
    s.update_parameters({"PPFD": intensity})
    s.simulate(300)
    s.update_parameter("PPFD", 5.0)
    s.simulate(500, steps=200)
    s.update_parameter("PPFD", intensity)
    s.simulate(800, steps=300)

    return s.get_result().unwrap_or_err().get_combined()


PhosphoMAX = 15

fig, axs = plt.subplots(3, 1, sharex=True)
plt.rcParams["figure.figsize"] = (18, 15)
for subplot in axs.flatten():
    subplot.set_xlim(200, 850)
    subplot.axvspan(300, 500, alpha=0.4, color="dimgray")
    subplot.set_facecolor("white")
    subplot.grid(color="lightgrey")
    subplot.tick_params(axis="both", which="major", labelsize=18)


axs[0].set_ylabel("Int. free [PO4$^{3−}$] (mM)", fontsize=14.0)
axs[0].text(
    245, 1.07, "L", fontsize=15, bbox={"facecolor": "w", "alpha": 0.5, "pad": 3}
)
axs[0].text(
    390, 1.07, "D", fontsize=15, bbox={"facecolor": "w", "alpha": 0.5, "pad": 3}
)
axs[0].text(
    590, 1.07, "L", fontsize=15, bbox={"facecolor": "w", "alpha": 0.5, "pad": 3}
)
axs[1].set_ylabel("TPT translocators (mM/s)", fontsize=14.0)
axs[2].set_ylabel("RuBisCO (mM/s)", fontsize=14.0)

axs[2].set_xlabel("Time (s)", fontsize=20.0)

col = {
    20: "royalblue",
    50: "forestgreen",
    100: "indianred",
    150: "darkorange",
    200: "black",
}


for i in [20, 50, 100, 150, 200]:
    print(i)

    df = LDL(get_matuszynska2019(), Tmax=500, intensity=i, darkphase=200.0)

    yvex1 = df["ex_pga"] + df["ex_gap"] + df["ex_dhap"]
    yrub1 = df["rubisco_carboxylase"]

    axs[0].plot(
        df.index,
        df["Orthophosphate"] / PhosphoMAX,
        color=col[i],
        linestyle="-.",
        linewidth=2.0,
        label=str(i) + "PFD",
        zorder=10.0,
    )
    axs[1].plot(
        df.index,
        yvex1,
        color=col[i],
        linestyle="-.",
        linewidth=2.0,
        label=str(i) + "PFD",
        zorder=10.0,
    )
    axs[2].plot(
        df.index,
        yrub1,
        color=col[i],
        linestyle="-.",
        linewidth=2.0,
        label=str(i) + "PFD",
        zorder=10.0,
    )

axs[1].legend(loc="best", bbox_to_anchor=(1.0, 0.7), prop={"size": 16})
plt.savefig("matuszynska2019-fig2.svg")
plt.show()